In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH150=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH150_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH150_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH150[mask]

from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 100 < x < 200]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=150, sigma=5, gamma=1, norm2=1, mu2=150, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (100, 200)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()

/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 46.13 (χ²/ndof = 1.1)      │              Nfcn = 844              │
│ EDM = 6.8e-05 (Goal: 0.0002)     │            time = 0.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.70    │   0.04    │            │            │         │         │       │
│ 1 │ mu     │  154.45   │   0.15    │            │            │   100   │   200   │       │
│ 2 │ sigma  │   7.83    │   0.29    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │    4.6    │    0.5    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.35    │   0.04    │            │            │         │         │       │
│ 5 │ mu2    │   144.2   │    1.1    │            │            │         │         │       │
│ 6 │ sigma2 │   19.3    │    0.6    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────┐
│        │    norm      mu   sigma   gamma   norm2     mu2  sigma2  gamma2 │
├────────┼─────────────────────────────────────────────────────────────────┤
│   norm │ 0.00198  0.0011 -0.0018  0.0199 -0.0017 -0.0470 -0.0176   0.000 │
│     mu │  0.0011  0.0224  -0.019   0.033 -0.0008  -0.048  -0.053   0.000 │
│  sigma │ -0.0018  -0.019  0.0815   -0.10  0.0010    0.03    0.10    0.00 │
│  gamma │  0.0199   0.033   -0.10   0.297 -0.0164   -0.46   -0.28     0.0 │
│  norm2 │ -0.0017 -0.0008  0.0010 -0.0164 0.00147  0.0406  0.0145  0.0000 │
│    mu2 │ -0.0470  -0.048    0.03   -0.46  0.0406    1.21    0.43     0.0 │
│ sigma2 │ -0.0176  -0.053    0.10   -0.28  0.0145    0.43   0.341    0.00 │
│ gamma2 │   0.000   0.000    0.00     0.0  0.0000     0.0    0.00       0 │
└────────┴─────────────────────────────────────────────────────────────────┘

In [2]:
fit_MH150_values={}
fit_MH150_errors={}

fit_values={'MH150': fit_MH150_values,}
fit_errors={'MH150_errors': fit_MH150_errors}



for param in m_voigt.parameters:
    fit_MH150_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH150_errors[error] = m_voigt.errors[error]

print(fit_MH150_values)
print(fit_MH150_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH150"]=fit_MH150_values
results["MH150_errors"]=fit_MH150_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH150"]=fit_MH150_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH150_errors"]=fit_MH150_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.7003234700431133, 'mu': 154.44907205736789, 'sigma': 7.829211636343578, 'gamma': 4.585004580556788, 'norm2': 0.3454107575352435, 'mu2': 144.24597939717438, 'sigma2': 19.2793459973479, 'gamma2': 0.001}
{'norm': 0.0444826193075926, 'mu': 0.14958046399544855, 'sigma': 0.2855149338668941, 'gamma': 0.5434984229595639, 'norm2': 0.03834500489781809, 'mu2': 1.1022375702654739, 'sigma2': 0.5840626731686642, 'gamma2': 1e-05}
